In [1]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec

In [2]:
load_dotenv()  # take environment variables from .env.

True

In [3]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
from transformers import pipeline

In [4]:
model_id = "dslim/bert-base-NER"


In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id) # huggingface model hub

In [6]:
ner_model = AutoModelForTokenClassification.from_pretrained(model_id)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
nlp = pipeline("ner", model=ner_model, tokenizer=tokenizer, device='cpu', aggregation_strategy="max")

In [8]:
from sentence_transformers import SentenceTransformer

In [10]:
retriver = SentenceTransformer('flax-sentence-embeddings/all_datasets_v3_mpnet-base')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\acer\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\acer\.cache\huggingface\hub\models--flax-sentence-embeddings--all_datasets_v3_mpnet-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
retriver

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [15]:
# create index in pinecone
# connect to pinecone and delete the index if it already exists
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)
pc.create_index(
    name="medium-posts",
    dimension=768,
    metric="cosine",
    spec = ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    )
)

IndexModel(name='medium-posts', metric='cosine', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), host='https://medium-posts-v29tsf3.svc.aped-4627-b74a.pinecone.io', private_host=None, vector_type='dense', dimension=768, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [16]:
idx = pc.index("medium-posts")

In [ ]:
# Obtain raw data
from datasets import load_dataset

dataset = load_dataset("fabiochiu/medium-articles", split="train")
dataset

DatasetNotFoundError: Dataset 'faboichu/medium-articles' doesn't exist on the Hub or cannot be accessed.

In [12]:
# prepare vector embeddings


In [13]:
# upsert operation